In [41]:
import pandas as pd
import glob


In [42]:
salario_minimo = pd.read_csv('salario_minimo_mensal_1940_2022.csv', sep=';')
salario_minimo_periodo = salario_minimo.loc[
    (salario_minimo.ano > 1949) &
    (salario_minimo.ano < 2026)
].copy()

salario_minimo_periodo[["unidade", "valor"]] = (
    salario_minimo_periodo["salario_minimo"]
    .str.split(" ", n=1, expand=True)
)

salario_minimo_periodo.rename(
    columns={
        'valor': 'valor_sal',
        'unidade': 'unidade_sal'
    },
    inplace=True
)

salario_minimo_periodo

,mes,ano,salario_minimo,unidade_sal,valor_sal
114,1,1950,"Cr$ 380,00",Cr$,"380,00"
115,2,1950,"Cr$ 380,00",Cr$,"380,00"
116,3,1950,"Cr$ 380,00",Cr$,"380,00"
117,4,1950,"Cr$ 380,00",Cr$,"380,00"
118,5,1950,"Cr$ 380,00",Cr$,"380,00"
...,...,...,...,...,...
1021,8,2025,"R$ 1518,00",R$,"1518,00"
1022,9,2025,"R$ 1518,00",R$,"1518,00"
1023,10,2025,"R$ 1518,00",R$,"1518,00"
1024,11,2025,"R$ 1518,00",R$,"1518,00"


In [43]:

path = "Batman/*.csv"

arquivos = glob.glob(path)

edicoes_batman = pd.concat([pd.read_csv(arq, sep=',') for arq in arquivos], ignore_index=True)

edicoes_batman[["unidade", "valor"]] = (
    edicoes_batman["preco_capa"]
    .str.split(" ", n=1, expand=True)
)

edicoes_batman.rename(
    columns={
        'valor': 'valor_gibi',
        'unidade': 'unidade_gibi'
    },
    inplace=True
)


edicoes_batman


,edicao_numero,mes,ano,preco_capa,numero_paginas,url,unidade_gibi,valor_gibi
0,1,3,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00"
1,2,4,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00"
2,3,5,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00"
3,4,6,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00"
4,5,7,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00"
...,...,...,...,...,...,...,...,...
751,32,9,2025,"R$ 31,90",84,http://www.guiadosquadrinhos.com/edicao/batman...,R$,"31,90"
752,1,10,2025,"R$ 14,90",52,http://www.guiadosquadrinhos.com/edicao/batman...,R$,"14,90"
753,2,12,2025,"R$ 19,90",52,http://www.guiadosquadrinhos.com/edicao/batman...,R$,"19,90"
754,3,1,2026,"R$ 19,90",52,http://www.guiadosquadrinhos.com/edicao/batman...,R$,"19,90"


In [44]:
data = pd.merge(edicoes_batman, salario_minimo_periodo, on=['mes', 'ano'])
data

,edicao_numero,mes,ano,preco_capa,numero_paginas,url,unidade_gibi,valor_gibi,salario_minimo,unidade_sal,valor_sal
0,1,3,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00","Cr$ 1200,00",Cr$,"1200,00"
1,2,4,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00","Cr$ 1200,00",Cr$,"1200,00"
2,3,5,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00","Cr$ 1200,00",Cr$,"1200,00"
3,4,6,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00","Cr$ 1200,00",Cr$,"1200,00"
4,5,7,1953,"Cr$ 3,00",36,http://www.guiadosquadrinhos.com/edicao/batman...,Cr$,"3,00","Cr$ 1200,00",Cr$,"1200,00"
...,...,...,...,...,...,...,...,...,...,...,...
749,30,7,2025,"R$ 31,90",84,http://www.guiadosquadrinhos.com/edicao/batman...,R$,"31,90","R$ 1518,00",R$,"1518,00"
750,31,8,2025,"R$ 31,90",92,http://www.guiadosquadrinhos.com/edicao/batman...,R$,"31,90","R$ 1518,00",R$,"1518,00"
751,32,9,2025,"R$ 31,90",84,http://www.guiadosquadrinhos.com/edicao/batman...,R$,"31,90","R$ 1518,00",R$,"1518,00"
752,1,10,2025,"R$ 14,90",52,http://www.guiadosquadrinhos.com/edicao/batman...,R$,"14,90","R$ 1518,00",R$,"1518,00"


In [45]:
data["valor_gibi_num"] = pd.to_numeric(
    data["valor_gibi"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)
data["valor_sal_num"] = pd.to_numeric(
    data["valor_sal"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)


data["%_sal_min"] = (data["valor_gibi_num"] / data["valor_sal_num"]) * 100

In [54]:
data["preço_pagina"] = (data["valor_sal_num"] / data["numero_paginas"])

In [55]:
data["data"] = pd.to_datetime(
    dict(year=data["ano"], month=data["mes"], day=1)
)

data = data.sort_values("data")

data.describe()

,edicao_numero,mes,ano,numero_paginas,valor_gibi_num,valor_sal_num,%_sal_min,data,preço_pagina
count,754.000000,754.000000,754.000000,754.000000,636.000000,753.000000,635.000000,754,753.000000
mean,36.184350,6.507958,1989.840849,65.591512,78.868789,8465.703147,1.372057,1990-04-19 16:29:17.029177728,182.742447
min,0.000000,1.000000,1953.000000,36.000000,0.300000,70.000000,0.105263,1953-03-01 00:00:00,0.426829
25%,13.000000,4.000000,1968.000000,36.000000,5.000000,268.800000,0.352734,1968-11-08 12:00:00,4.150000
50%,28.000000,6.500000,1992.000000,52.000000,7.500000,880.000000,1.106195,1992-05-16 12:00:00,12.120000
75%,55.000000,9.750000,2010.000000,84.000000,15.000000,2400.000000,1.967857,2010-09-23 12:00:00,66.666667
max,114.000000,12.000000,2025.000000,164.000000,4700.000000,230000.000000,6.556291,2025-12-01 00:00:00,4423.076923
std,28.725134,3.451130,22.764190,32.114879,377.462826,25274.749447,1.201618,NaN,507.413969


In [56]:
import plotly.express as px

fig = px.line(
    data,
    x="data",
    y="%_sal_min",
    title="% do Salário Mínimo ao Longo do Tempo"
)

fig.update_layout(
    xaxis_title="Ano",
    yaxis_title="% do Salário Mínimo"
)

fig.show()

In [57]:
import plotly.express as px

fig = px.line(
    data,
    x="data",
    y="preço_pagina",
    title="'Preço da Página' ao Longo do Tempo"
)

fig.update_layout(
    xaxis_title="Ano",
    yaxis_title="preço pagina"
)

fig.show()

In [48]:
import pandas as pd

# Remove NaN
df2 = data#.dropna(subset=['%_sal_min']).copy()

# =========================
# MÁXIMO
# =========================
i_max = df2['%_sal_min'].idxmax()
max_row = df2.loc[i_max]

# =========================
# MÍNIMO
# =========================
i_min = df2['%_sal_min'].idxmin()
min_row = df2.loc[i_min]

# =========================
# MÉDIA E MEDIANA
# =========================
media = df2['%_sal_min'].mean()
mediana = df2['%_sal_min'].median()

# =========================
# PRINT ORGANIZADO
# =========================
print("📈 MAIOR CUSTO RELATIVO:")
print(
    f"Data: {max_row['data']:%d/%m/%Y} | "
    f"Edição: {max_row['edicao_numero']} | "
    f"{max_row['%_sal_min']:.2f}% do salário mínimo "
    f"(Gibi={max_row['valor_gibi_num']:.2f}, "
    f"Salário={max_row['valor_sal_num']:.2f})"
)

print("\n📉 MENOR CUSTO RELATIVO:")
print(
    f"Data: {min_row['data']:%d/%m/%Y} | "
    f"Edição: {min_row['edicao_numero']} | "
    f"{min_row['%_sal_min']:.4f}% do salário mínimo "
    f"(Gibi={min_row['valor_gibi_num']:.2f}, "
    f"Salário={min_row['valor_sal_num']:.2f})"
)

print("\n📊 ESTATÍSTICAS GERAIS:")
print(f"Média: {media:.3f}% do salário mínimo")
print(f"Mediana: {mediana:.3f}% do salário mínimo")

📈 MAIOR CUSTO RELATIVO:
Data: 01/08/2000 | Edição: 1 | 6.56% do salário mínimo (Gibi=9.90, Salário=151.00)

📉 MENOR CUSTO RELATIVO:
Data: 01/10/1957 | Edição: 56 | 0.1053% do salário mínimo (Gibi=4.00, Salário=3800.00)

📊 ESTATÍSTICAS GERAIS:
Média: 1.372% do salário mínimo
Mediana: 1.106% do salário mínimo


O pico no ano 2000 se deve a sexta série de Batman pela Abril, conhecida como *Super-Heróis Premium*, uma iniciativa da Editora Abril que transformou os formatinhos em formato americano e com maior número de páginas.